# Project-LUCID · Colab Ollama (T4)

Colab에서 Ollama를 시작하고 Cloudflare Quick Tunnel로 로컬 LUCID에 연결하는 노트북이야.
위에서부터 순서대로 실행해. 텍스트 모델은 `qwen2.5:3b`, 비전 모델은 `gemma3:12b`를 사용해.
비전 기능이 필요 없으면 모델 설정 셀에서 `ENABLE_VISION_MODEL = False`로 바꿔 시작 시간을 줄일 수 있어.

## 1. 런타임 GPU 확인

In [ ]:
!nvidia-smi

## 2. Ollama 설치 및 서비스 시작

In [ ]:
import subprocess

subprocess.run(
    ["apt-get", "install", "-y", "--no-install-recommends", "zstd"],
    check=True,
)

In [ ]:
import shutil
import subprocess

if shutil.which("ollama") is None:
    install_script = "set -o pipefail && curl -fsSL https://ollama.com/install.sh | sh"
    subprocess.run(["bash", "-lc", install_script], check=True)
else:
    print("Ollama가 이미 설치되어 있어.")

subprocess.run(["ollama", "--version"], check=True)


In [ ]:
import os
import subprocess
import time
from urllib.error import URLError
from urllib.request import urlopen

OLLAMA_API = "http://127.0.0.1:11434"
_ollama_process = globals().get("_ollama_process")

def _ollama_is_ready():
    try:
        with urlopen(f"{OLLAMA_API}/api/tags", timeout=2) as response:
            return response.status == 200
    except (OSError, URLError, TimeoutError):
        return False

if _ollama_is_ready():
    print("Ollama가 이미 준비되어 있어.")
else:
    if _ollama_process is None or _ollama_process.poll() is not None:
        ollama_env = os.environ.copy()
        ollama_env["OLLAMA_HOST"] = "0.0.0.0:11434"
        _ollama_log = open("/content/ollama.log", "a", encoding="utf-8")
        _ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            env=ollama_env,
            stdout=_ollama_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        globals()["_ollama_process"] = _ollama_process
        globals()["_ollama_log"] = _ollama_log

    deadline = time.monotonic() + 60
    while time.monotonic() < deadline:
        if _ollama_is_ready():
            print("Ollama API 준비 완료: http://127.0.0.1:11434")
            break
        if _ollama_process.poll() is not None:
            break
        time.sleep(1)
    else:
        raise TimeoutError("Ollama가 60초 안에 준비되지 않았어. /content/ollama.log를 확인해줘.")

    if not _ollama_is_ready():
        log_path = "/content/ollama.log"
        try:
            with open(log_path, encoding="utf-8", errors="replace") as log_file:
                log_tail = "".join(log_file.readlines()[-30:])
        except OSError:
            log_tail = "로그 파일을 읽을 수 없어."
        raise RuntimeError(f"Ollama 시작 실패. 로그 마지막 부분:\n{log_tail}")

## 3. LUCID 모델 설정 및 다운로드

In [ ]:
import subprocess
from urllib.request import urlopen
import json

TEXT_MODEL = "qwen2.5:3b"
VISION_MODEL = "gemma3:12b"
ENABLE_VISION_MODEL = True

models_to_pull = [("text", TEXT_MODEL)]
if ENABLE_VISION_MODEL:
    models_to_pull.append(("vision", VISION_MODEL))

with urlopen(f"{OLLAMA_API}/api/tags", timeout=10) as response:
    installed_names = {model["name"] for model in json.load(response).get("models", [])}

for role, model_name in models_to_pull:
    if model_name in installed_names:
        print(f"[{role}] {model_name} 이미 설치되어 있어. 건너뜀.")
        continue
    print(f"\n[{role}] {model_name} 다운로드")
    subprocess.run(["ollama", "pull", model_name], check=True)
    installed_names.add(model_name)

subprocess.run(["ollama", "list"], check=True)


## 4. Ollama API 확인

In [ ]:
import json
from urllib.request import urlopen

with urlopen(f"{OLLAMA_API}/api/tags", timeout=10) as response:
    model_data = json.load(response)

print("Ollama API 응답 정상")
print("설치된 모델:", ", ".join(model["name"] for model in model_data.get("models", [])))

## 5. Cloudflare Quick Tunnel 시작

In [ ]:
from pathlib import Path
import subprocess

cloudflared = Path("/usr/local/bin/cloudflared")
if not cloudflared.exists() or cloudflared.stat().st_size == 0:
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", str(cloudflared),
        ],
        check=True,
    )
cloudflared.chmod(cloudflared.stat().st_mode | 0o111)
subprocess.run([str(cloudflared), "--version"], check=True)

아래 셀은 터널을 계속 실행해. 출력에 표시되는 `trycloudflare.com` 주소를 LUCID의 Ollama 주소로 사용해.
Quick Tunnel 주소는 Colab 세션마다 바뀔 수 있어.

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:11434